# Sound-source presence and patterns

Run from the repository root after building the database from the prepared data bundle; see README.md.


# Initialization and database connection

In [ ]:
from analysis_paths import connect_analysis, data_root, output_path, require_file
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from helpers import plot_sensors_on_map_df
from helpers import plot_source_avg_per_minute

# Connect to the prepared DuckDB database
db = connect_analysis()


print("DuckDB version:", db.execute("SELECT version()" ).fetchone()[0])

# Total sound event ratios

In [ ]:
labels = ["vehicle", "honking", "aircraft", "siren", "human", "bark", "bird", "church", "music", "wind", "rain"]
labels_with_events = [f"{label}_events" for label in labels]

event_cols_sql = ",\n    ".join(
  [f"SUM(({lbl} > 0.5)::INTEGER) AS {lbl}_events" for lbl in labels]
)

# For all events
sum_events = db.execute(f"""
  SELECT
  sensor_id,
  {event_cols_sql},
  COUNT(*) AS total_events
  FROM sources
  GROUP BY sensor_id
""").df()

sum_events[labels_with_events] = sum_events[labels_with_events].div(sum_events["total_events"], axis=0)
sum_events = sum_events.drop(["sensor_id", "total_events"], axis=1).multiply(100).mean().sort_values(ascending=False).to_frame().T
sum_events = sum_events.rename(columns={f"{label}_events": label for label in labels})

# For events above 60 dB(A)
sum_events_60 = db.execute(f"""
  SELECT
  sensor_id,
  {event_cols_sql},
  COUNT(*) AS total_events
  FROM sources
  WHERE avg_dba > 60
  GROUP BY sensor_id
""").df()

sum_events_60[labels_with_events] = sum_events_60[labels_with_events].div(sum_events_60["total_events"], axis=0)
sum_events_60 = sum_events_60.drop(["sensor_id", "total_events"], axis=1).multiply(100).mean().sort_values(ascending=False).to_frame().T
sum_events_60 = sum_events_60.rename(columns={f"{label}_events": label for label in labels})

# Merge all and >60dBA results together
plot_df = pd.DataFrame({"all": sum_events.iloc[0], "dB(A) > 60": sum_events_60.iloc[0]})
plot_df = plot_df.loc[sum_events.columns]

fig, ax = plt.subplots(figsize=(6, 3), dpi=200)
plot_df.plot(kind='bar', ax=ax)
#   ax.bar_label(container, fmt="%.1f", padding=3)
ax.set_ylim(0, float(plot_df.to_numpy().max()) * 1.15)
ax.set_xlabel('Sound Event')
ax.set_ylabel('Average Presence Ratio (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

# Sound event presence per category

In [ ]:
def show_sound_event_presence(db, label, where_clause="", binnenstad_overlay=False, title=False):
  top_pct = db.sql(f"""
    SELECT
      sensor_id,
      100.0 * AVG(({label} > 0.5)::DOUBLE) AS pct
    FROM sources
    {where_clause}
    GROUP BY sensor_id
    ORDER BY pct ASC
  """).df().rename(columns={'pct': f'pct_{label}'})

  plot_sensors_on_map_df(
    top_pct,
    color_col=f'pct_{label}',
    title=f"Sensors by % {label.capitalize()} (prob > 0.5{(' and ' + where_clause.strip() if where_clause else '')})" if title else None,
    outline=True,
    binnenstad_polygon=binnenstad_overlay,
  )

## Vehicle ratios

In [ ]:
show_sound_event_presence(db, 'vehicle', where_clause="WHERE avg_dba > 60", title=False)

## Honking ratios

In [ ]:
show_sound_event_presence(db, 'honking', title=False)

## Aircraft ratios

In [ ]:
show_sound_event_presence(db, 'aircraft')

## Siren ratios

In [ ]:
show_sound_event_presence(db, 'siren', title=False)

## Human noise ratios

In [ ]:
show_sound_event_presence(db, 'human')

## Dog bark ratios

In [ ]:
show_sound_event_presence(db, 'bark')

## Bird song ratios

In [ ]:
show_sound_event_presence(db, 'bird')

## Church bell ratios

In [ ]:
show_sound_event_presence(db, 'church', binnenstad_overlay=True)

## Music ratios

In [ ]:
show_sound_event_presence(db, 'music')

## Wind ratios

In [ ]:
show_sound_event_presence(db, 'wind')

## Rain ratios

In [ ]:
show_sound_event_presence(db, 'rain')

# Sound event loudness

In [ ]:
# Query average loudness per source type using proper linear averaging (SPL_A), then convert to dBA in SQL
THRESH = 0.5
source_types = [
  'vehicle', 'honking', 'aircraft', 'siren', 'human', 'bark', 'bird', 'church', 'music', 'wind', 'rain'
]

# Build SQL: average in linear domain then convert to dBA within DuckDB
select_parts = [
    (
        f"SELECT '{src}' AS source, AVG(avg_spl_a) AS avg_spl_a "
        f"FROM sources WHERE {src} > {THRESH} AND avg_spl_a IS NOT NULL "
    )
    for src in source_types
]
sql = (
    "WITH agg AS (\n  "
    + "\n  UNION ALL\n  ".join(select_parts)
    + "\n)\n"
    + "SELECT source, avg_spl_a, 10*log10(avg_spl_a) AS avg_dba FROM agg ORDER BY avg_dba DESC"
)

avg_spl_per_source = db.sql(sql).df()

plt.figure(figsize=(10, 5))
sns.barplot(data=avg_spl_per_source, x='source', y='avg_dba')
plt.ylabel('Average Loudness (dBA)')
plt.xlabel('Source Type')
plt.title(f'Average Loudness per Source Type (prob > {THRESH})')
plt.xticks(rotation=30)
plt.ylim(40, 80)
plt.tight_layout()
plt.show()

# Diurnal sound event presence ratios

Presence (%) = 100 × detections / recorded fragments, pooled across sensors and dates for each minute. Detection uses score > cutoff. All timestamped fragments enter the denominator, including NULL scores (none in the verified data), which are not positive detections. Observed zero minutes remain zero; absent minutes are not filled. The dotted line averages minute percentages over a centred 60-minute window (minimum 20 points, truncated at day boundaries). Legacy count columns and detection-conditioned loudness are retained for comparison.

Run python scripts/regenerate_diurnal_plots.py to export CSVs and figures without changing this notebook; execute these cells to refresh inline outputs. New exports and verification are in out/diurnal_presence/ and figures/diurnal_presence/.

## Vehicles

In [ ]:
SOURCE = 'vehicle'
THRESH = 0.5
fig, vehicle_diurnal = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    return_df=True,
    window_size=60,
    plot_median_dba=True,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Honking

In [ ]:
SOURCE = 'honking'
THRESH = 0.5
fig = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    window_size=60,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Aircraft

In [ ]:
SOURCE = 'aircraft'
THRESH = 0.5
fig = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    window_size=60,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Sirens

In [ ]:
SOURCE = 'siren'
THRESH = 0.5
fig = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    window_size=60,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Crowd noise

In [ ]:
SOURCE = 'human'
THRESH = 0.5
fig, human_diurnal = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    return_df=True,
    window_size=60,
    plot_median_dba=False,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Music

In [ ]:
SOURCE = 'music'
THRESH = 0.5
fig = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    window_size=60,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Bird song

In [ ]:
SOURCE = 'bird'
THRESH = 0.5
fig, bird_diurnal = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    return_df=True,
    window_size=60,
    plot_median_dba=False,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Church bell ringing

In [ ]:
SOURCE = 'church'
THRESH = 0.5
fig, church_diurnal = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    return_df=True,
    window_size=60,
    plot_median_dba=False,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Dog bark

In [ ]:
SOURCE = 'bark'
THRESH = 0.5
fig = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    window_size=60,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Wind

In [ ]:
SOURCE = 'wind'
THRESH = 0.5
fig = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    window_size=60,
    height=300,
    width=500,
    rangeslider_visible=False,
)

## Rain

In [ ]:
SOURCE = 'rain'
THRESH = 0.5
fig = plot_source_avg_per_minute(
    SOURCE,
    threshold=THRESH,
    window_size=60,
    height=300,
    width=500,
    rangeslider_visible=False,
)

# Median loudness of air raid sirene

In [ ]:
from numpy import maximum


air_raid_start = pd.to_datetime("2025-06-02 11:59:00")
air_raid_end = pd.to_datetime("2025-06-02 16:03:00")

siren_events_air_raid = db.sql(f"""
  SELECT
    sensor_id,
    time,
    avg_spl_a,
    max_spl_a,
    siren
  FROM sources
  WHERE siren > 0.5
    AND time >= '{air_raid_start}'
    AND time <= '{air_raid_end}'
""").df()

median__siren_dba = 10 * np.log10(siren_events_air_raid['avg_spl_a'].mean())
print(f"Median siren dBA during air raid: {median__siren_dba:.1f} dBA")

maximum_siren_dba = 10 * np.log10(siren_events_air_raid['max_spl_a'].max())
print(f"Maximum siren dBA during air raid: {maximum_siren_dba:.2f} dBA")

# Sound event co-occurance

In [ ]:
# Conditional source co-occurrence matrix: P(j | i) = probability that source j is present given source i is present
# Rows (y-axis): condition/source i ("given" source)
# Columns (x-axis): target/source j ("result" source)

source_cols = [
    'vehicle', 'honking', 'aircraft', 'siren', 'human', 'bark', 'bird', 'church', 'music', 'wind', 'rain'
]

# Aggregate joint counts in DuckDB; same strict threshold and denominators.
from helpers import source_cooccurrence
cond_df = source_cooccurrence(db, source_cols, threshold=0.5)

fig = px.imshow(
    cond_df,
    text_auto='.2f',
    color_continuous_scale='Blues',
    aspect='auto',
    title='Conditional Source Co-occurrence: P(j | i)'
)
fig.update_layout(
    height=550,
    width=750,
    xaxis_title='Target j',
    yaxis_title='Condition i',
)
fig.show()